# 04. Machine Learning Starter Framework: Diagnosis Classification

## Objective
Set up a reproducible, leakage-free classification workflow to classify **existing diabetes diagnosis** (`Diagnosis` = 0 or 1).

> **Critical Clinical & Analytical Disclaimer:** Because `diabetes_data.csv` captures existing patient records rather than longitudinal follow-up, this framework classifies **current diagnosis status**—it does **not** predict future diabetes risk.

## Workflow Plan
1. Load and clean data using `src/data/preprocessing.py`.
2. Split into stratified 80/20 train/test sets **before** fitting scalers or encoders.
3. Compare a naive baseline (`DummyClassifier`) against `LogisticRegression`, `RandomForestClassifier`, and `GradientBoostingClassifier` using 5-fold Stratified Cross-Validation.
4. Run a sensitivity comparison excluding diagnostic/treatment columns (`FastingBloodSugar`, `HbA1c`, `AntidiabeticMedications`).
5. Evaluate held-out test metrics (Accuracy, Precision, Recall, F1, ROC-AUC, PR-AUC) and inspect confusion matrices.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.models.train import run_training_framework
from src.models.evaluate import evaluate_all_models, save_confusion_matrix, save_metrics_report

artifacts_full = run_training_framework(exclude_diagnostic_markers=False, random_state=42)
artifacts_full.cv_results

### Analysis: Cross-Validation Results (Full Feature Set)
Compare each candidate model against the `dummy_majority` baseline across F1-score, Recall (critical for minimizing missed positive cases), ROC-AUC, and PR-AUC. Next, evaluate the fitted pipelines on the held-out test split.

In [ ]:
test_metrics_full = evaluate_all_models(
    artifacts_full.pipelines,
    artifacts_full.X_test,
    artifacts_full.y_test,
)
best_model_name = test_metrics_full.iloc[0]["model"]
save_confusion_matrix(
    artifacts_full.pipelines[best_model_name],
    artifacts_full.X_test,
    artifacts_full.y_test,
    model_name=best_model_name,
)
test_metrics_full

### Analysis: Sensitivity Experiment (Excluding Diagnostic Biomarkers & Medications)
Because `HbA1c`, `FastingBloodSugar`, and `AntidiabeticMedications` directly reflect diagnostic criteria or active diabetes treatment, we also evaluate how well demographic, lifestyle, symptom, and comorbidity features classify `Diagnosis` on their own.

In [ ]:
artifacts_no_diag = run_training_framework(exclude_diagnostic_markers=True, random_state=42)
test_metrics_no_diag = evaluate_all_models(
    artifacts_no_diag.pipelines,
    artifacts_no_diag.X_test,
    artifacts_no_diag.y_test,
)
test_metrics_no_diag

### Summary & Next Modeling Steps
- Record final model comparison results only after executing this notebook end-to-end.
- Compare the drop in F1 / ROC-AUC when diagnostic labs and medications are removed to quantify reliance on direct diagnostic criteria.
- Future work (Stage 2): hyperparameter tuning, probability calibration curves, slice-based error analysis across age/gender/socioeconomic groups, and SHAP feature-importance explanations.